# aa2codon v2: focused hyperparameter tuning
Five parameters, five values each, one coordinate sweep (at most 21 unique trials). Use a GPU runtime. Completed trials are stored in Google Drive; a disconnected trial restarts, not the whole search. Validation NLL excludes deterministic codons. Calibration and test data are not used in tuning.

In [ ]:
!git clone -b pytorch-xielu-uncertainty https://github.com/pyypyyy/aa2codon.git /content/aa2codon_v2
%cd /content/aa2codon_v2/pytorch_v2
!pip -q install biopython plotly
from google.colab import drive
drive.mount('/content/drive')
import torch
print(torch.__version__, torch.cuda.is_available())

In [ ]:
from pathlib import Path
print(*[str(p) for p in Path('../Genomes').rglob('*') if p.is_file()], sep='\n')
FASTA = [] # Set CDS FASTA paths for one host; same paths for final training.
TUNING_OUT = '/content/drive/MyDrive/aa2codon_v2/tuning'
BATCH = 16 # Fixed across trials; lower if GPU memory is insufficient.
MAX_LENGTH = 1024
EPOCHS = 8
PRUNE = True
assert FASTA, 'Set FASTA to your chosen CDS file(s).'

In [ ]:
from tune import tune, plot_trials
best, trials = tune(FASTA, TUNING_OUT, batch=BATCH, max_length=MAX_LENGTH, epochs=EPOCHS, prune=PRUNE)
print(best['config'], 'lr:', best['lr'], 'weight_decay:', best['weight_decay'])
print('Validation NLL:', best['validation_nll'])
plot_trials(trials).show()

In [ ]:
import pandas as pd
pd.DataFrame([{**t['params'], 'nll': t['score'], 'status': t['status'], 'seconds': t['seconds'], 'parameters': t['parameters']} for t in trials]).sort_values('nll')